In [1]:
# ── Librerías ─────────────────────────────────────────────────────────
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import janitor
import sqlalchemy as sa
import os
from pathlib import Path
from dotenv import load_dotenv

%matplotlib inline

# ── Opciones de visualización ─────────────────────────────────────────
# Desactivar notación científica
pd.set_option('display.float_format', '{:.3f}'.format)
np.set_printoptions(suppress=True)

# Ver todas las columnas al imprimir un DataFrame
pd.set_option('display.max_columns', None)

# ── Rutas del proyecto ────────────────────────────────────────────────
# Se resuelven desde la ubicación del notebook, así funcionan
# igual en cualquier máquina
RAIZ = Path.cwd().parent

ORIGINALES   = RAIZ / '02_datos' / '01_Originales'
VALIDACION   = RAIZ / '02_datos' / '02_Validacion'
ENTRENAMIENTO = RAIZ / '02_datos' / '03_Entrenamiento'
CACHES       = RAIZ / '02_datos' / '04_Caches'
MODELOS      = RAIZ / '05_modelos'
RESULTADOS   = RAIZ / '06_resultados'

# ── Variables de entorno ──────────────────────────────────────────────
load_dotenv(RAIZ / '.env')

print('✅ Entorno listo')
print(f'   Raíz del proyecto: {RAIZ}')

✅ Entorno listo
   Raíz del proyecto: c:\Users\Admin\Desktop\Desktop\CarpetaFisica\Data Detective\Ciencia_Datos_2\Caso_Agentes


In [2]:
# ── Carga del tablón de entrenamiento actual ─────────────────────────
df = pd.read_pickle(ENTRENAMIENTO / '05_train_tablon_preseleccion.pkl')

print(f'✅ Tablón cargado')
print(f'   Shape: {df.shape[0]:,} filas x {df.shape[1]} columnas')
print(f'\nColumnas:')
for col in df.columns:
    print(f'   - {col}')


✅ Tablón cargado
   Shape: 28,015 filas x 13 columnas

Columnas:
   - trabajo_student
   - impago_unknown
   - prestamo_hipotecario_yes
   - canal_de_contacto_telephone
   - mes_sin
   - mes_cos
   - resultado_campana_anterior_nonexistent
   - resultado_campana_anterior_success
   - edad_ss
   - num_contactos_esta_campana_log_ss
   - variacion_tasa_empleo_ss
   - euribor3m_ss
   - contrata_fondos


In [3]:
# ── Diagnóstico de desbalanceo de la target ─────────────────────────
TARGET = 'contrata_fondos'
y = df[TARGET]

N_total = len(df)
conteo = y.value_counts().sort_index()
proporcion = (conteo / N_total * 100).round(2)

# Clase mayoritaria / minoritaria
clase_mayoritaria = conteo.idxmax()
clase_minoritaria = conteo.idxmin()
ratio_may_min = conteo[clase_mayoritaria] / conteo[clase_minoritaria]

tabla_desbalanceo = pd.DataFrame({
    'n_obs': conteo,
    'proporcion_%': proporcion
})
tabla_desbalanceo.index.name = TARGET

print(f'N total: {N_total:,} filas')
print(f'\nDistribución de clases de {TARGET}:\n')
print(tabla_desbalanceo.to_string())
print(f'\nRatio mayoría/minoría: {ratio_may_min:.2f}:1')


N total: 28,015 filas

Distribución de clases de contrata_fondos:

                 n_obs  proporcion_%
contrata_fondos                     
0                24781        88.460
1                 3234        11.540

Ratio mayoría/minoría: 7.66:1


In [4]:
# ── Split interno train/test para experimentos (solo A_06) ───────────
from sklearn.model_selection import train_test_split

RANDOM_STATE = 42
TEST_SIZE = 0.2

X = df.drop(columns=[TARGET])
y = df[TARGET]

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
    stratify=y
)

print(f'X_train: {X_train.shape} | y_train: {y_train.shape}')
print(f'X_test : {X_test.shape} | y_test : {y_test.shape}')

# ── Distribución de clases en train y test ───────────────────────────
def _tabla_clases(serie):
    conteo_serie = serie.value_counts().sort_index()
    prop_serie = (conteo_serie / len(serie) * 100).round(2)
    tabla = pd.DataFrame({'n': conteo_serie, '%': prop_serie})
    return tabla

print(f'\nDistribución en TRAIN ({TARGET}):')
print(_tabla_clases(y_train).to_string())
print(f'\nDistribución en TEST ({TARGET}):')
print(_tabla_clases(y_test).to_string())


X_train: (22412, 12) | y_train: (22412,)
X_test : (5603, 12) | y_test : (5603,)

Distribución en TRAIN (contrata_fondos):
                     n      %
contrata_fondos              
0                19825 88.460
1                 2587 11.540

Distribución en TEST (contrata_fondos):
                    n      %
contrata_fondos             
0                4956 88.450
1                 647 11.550


In [6]:
# ── Comprobación de dependencias (imbalanced-learn) ──────────────────
try:
    from imblearn.under_sampling import RandomUnderSampler
    from imblearn.over_sampling import RandomOverSampler
    from imblearn.combine import SMOTETomek
    print('✅ imbalanced-learn disponible')
except ImportError as exc:
    print('❌ imbalanced-learn NO está instalado')
    print('   → Ejecuta en terminal:  uv add imbalanced-learn')


✅ imbalanced-learn disponible


In [7]:
# ── Ejecución de TODOS los escenarios de balanceo ────────────────────
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from imblearn.under_sampling import RandomUnderSampler
from imblearn.over_sampling import RandomOverSampler
from imblearn.combine import SMOTETomek


def _entrenar_logit_auc(X_tr, y_tr, X_te, y_te):
    """Entrena regresión logística y devuelve AUC sobre test."""
    modelo = LogisticRegression(max_iter=1000, random_state=RANDOM_STATE)
    modelo.fit(X_tr, y_tr)
    prob_pos = modelo.predict_proba(X_te)[:, 1]
    auc = roc_auc_score(y_te, prob_pos)
    return auc


def _resumen_train(X_tr, y_tr):
    """Devuelve (n_train, n_positivos, %positivos)."""
    conteo_serie = y_tr.value_counts().sort_index()
    n_train = len(y_tr)
    n_pos = int(conteo_serie.loc[1])
    pct_pos = (n_pos / n_train * 100)
    return n_train, n_pos, pct_pos


escenarios = [
    {'nombre': 'Sin balanceo', 'sampler': None},
    {'nombre': 'RandomUnderSampler',
     'sampler': RandomUnderSampler(random_state=RANDOM_STATE)},
    {'nombre': 'RandomOverSampler',
     'sampler': RandomOverSampler(random_state=RANDOM_STATE)},
    {'nombre': 'SMOTETomek',
     'sampler': SMOTETomek(random_state=RANDOM_STATE)},
]

resultados = []
for esc in escenarios:
    print(f'⏳ Ejecutando: {esc["nombre"]} ...')
    X_tr = X_train
    y_tr = y_train
    if esc['sampler'] is not None:
        X_tr, y_tr = esc['sampler'].fit_resample(X_train, y_train)

    auc = _entrenar_logit_auc(X_tr, y_tr, X_test, y_test)
    n_train, n_pos, pct_pos = _resumen_train(X_tr, y_tr)

    resultados.append({
        'escenario': esc['nombre'],
        'metodo_sampler': esc['nombre'],
        'n_train_post_balanceo': n_train,
        'n_positivos_train': n_pos,
        'pct_clase_positiva_train': pct_pos,
        'AUC_test': auc,
    })
    print(f'   ✅ AUC test = {auc:.4f} | '
          f'n_train = {n_train:,} | % positivos train = {pct_pos:.2f}%')

print('\n✅ Todos los escenarios ejecutados.')


⏳ Ejecutando: Sin balanceo ...
   ✅ AUC test = 0.7772 | n_train = 22,412 | % positivos train = 11.54%
⏳ Ejecutando: RandomUnderSampler ...
   ✅ AUC test = 0.7755 | n_train = 5,174 | % positivos train = 50.00%
⏳ Ejecutando: RandomOverSampler ...
   ✅ AUC test = 0.7754 | n_train = 39,650 | % positivos train = 50.00%
⏳ Ejecutando: SMOTETomek ...
   ✅ AUC test = 0.7748 | n_train = 38,684 | % positivos train = 50.00%

✅ Todos los escenarios ejecutados.


In [8]:
# ── Tabla comparativa de escenarios ──────────────────────────────────
comentarios = {
    'Sin balanceo': 'Referencia. Mantiene distribución original.',
    'RandomUnderSampler': 'Descarta ~77% de las filas. Pérdida de información.',
    'RandomOverSampler': 'Duplica minoritarios. Aumenta tamaño train.',
    'SMOTETomek': 'Sintéticos + limpieza. Complejidad y coste extra.',
}

# Añadir comentarios con bucle tradicional (convención del proyecto)
lista_comentarios = []
for esc in resultados:
    lista_comentarios.append(comentarios.get(esc['escenario'], ''))

tabla_comparativa = pd.DataFrame(resultados)
tabla_comparativa['AUC_test'] = tabla_comparativa['AUC_test'].round(4)
tabla_comparativa['pct_clase_positiva_train'] = (
    tabla_comparativa['pct_clase_positiva_train'].round(2)
)
tabla_comparativa['comentarios'] = lista_comentarios
tabla_comparativa = tabla_comparativa.sort_values(
    'AUC_test', ascending=False
).reset_index(drop=True)

print('Tabla comparativa de escenarios (ordenada por AUC desc):\n')
print(tabla_comparativa.to_string(index=False))


Tabla comparativa de escenarios (ordenada por AUC desc):

         escenario     metodo_sampler  n_train_post_balanceo  n_positivos_train  pct_clase_positiva_train  AUC_test                                         comentarios
      Sin balanceo       Sin balanceo                  22412               2587                    11.540     0.777         Referencia. Mantiene distribución original.
RandomUnderSampler RandomUnderSampler                   5174               2587                    50.000     0.775 Descarta ~77% de las filas. Pérdida de información.
 RandomOverSampler  RandomOverSampler                  39650              19825                    50.000     0.775         Duplica minoritarios. Aumenta tamaño train.
        SMOTETomek         SMOTETomek                  38684              19342                    50.000     0.775   Sintéticos + limpieza. Complejidad y coste extra.


In [9]:
# ── Documentación: decisión de NO balancear ─────────────────────────
from datetime import date

CARPETA_BALANCEO_RES = RESULTADOS / 'Balanceo'
CARPETA_BALANCEO_DOC = RAIZ / '01_Documentos' / 'Balanceo'
CARPETA_BALANCEO_RES.mkdir(parents=True, exist_ok=True)
CARPETA_BALANCEO_DOC.mkdir(parents=True, exist_ok=True)

# Tabla comparativa en formato Markdown (orden ya descendente por AUC)
lineas_tabla = []
lineas_tabla.append('| Escenario | n_train post-balanceo | % clase positiva train | AUC test |')
lineas_tabla.append('|---|---|---|---|')
for _, fila in tabla_comparativa.iterrows():
    linea = (f'| {fila["escenario"]} | {int(fila["n_train_post_balanceo"]):,} | '
             f'{fila["pct_clase_positiva_train"]:.2f}% | '
             f'{fila["AUC_test"]:.4f} |')
    lineas_tabla.append(linea)
tabla_md = '\n'.join(lineas_tabla)

n_0 = int(conteo.loc[0])
n_1 = int(conteo.loc[1])
pct_0 = proporcion.loc[0]
pct_1 = proporcion.loc[1]

informe_balanceo = f"""# Informe de Balanceo de Clases

- **Fecha**: {date.today().isoformat()}
- **Agente**: A_06_BalanceadorClases
- **Tipo de problema**: clasificación binaria
- **Target**: `{TARGET}`
- **Tablón de origen**: `../02_datos/03_Entrenamiento/05_train_tablon_preseleccion.pkl`

## 1. Diagnóstico de desbalanceo inicial

- **N total**: {N_total:,} filas.
- Clase 0 (no contrata): {n_0:,} filas ({pct_0:.2f}%).
- Clase 1 (contrata): {n_1:,} filas ({pct_1:.2f}%).
- **Ratio mayoría/minoría**: {ratio_may_min:.2f}:1.

Valoración: desbalanceo **moderado** (clase positiva ≈ 11,5%).

## 2. Metodología

- Dataset de trabajo: tablón de entrenamiento **completo** (sin muestrear).
- Split interno estratificado: {TEST_SIZE:.0%} test, `random_state={RANDOM_STATE}`.
- Modelo: regresión logística.
- Métrica: **ROC AUC** sobre el test interno sin balancear.
- Escenarios: sin balanceo, RandomUnderSampler, RandomOverSampler y SMOTETomek.
- El balanceo se aplicó **solo sobre el train interno**; test se mantuvo intacto.

## 3. Escenarios probados

{tabla_md}

## 4. Recomendación del agente

**No balancear.** Ningún método mejora el AUC del escenario base (0.7772):
- RandomUnderSampler: 0.7755 (descarta ~77% de las filas).
- RandomOverSampler: 0.7754 (duplica el tamaño del train sin beneficio).
- SMOTETomek: 0.7748 (mayor complejidad y coste, AUC más bajo).

La regresión logística mantiene buena discriminación con clases desbalanceadas,
y ROC AUC (independiente del umbral) no mejora con el balanceo.

## 5. Decisión final del usuario

**No aplicar balanceo.** No se genera un tablón balanceado. El dataframe de
entrenamiento se mantiene con su distribución original (tablón de A_05).

## 6. Impacto esperado en modelización

- Se trabajará con la distribución original (≈88,5% / 11,5%).
- El modelo seguirá siendo entrenado por máxima verosimilitud (buena calibración).
- Si en A_07 se prioriza recall/precisión con umbrales bajos sobre la clase
  positiva, se podrá reconsiderar el balanceo o ajustar umbral de decisión.
"""

escenarios_balanceo = f"""# Escenarios de Balanceo Probados

- **Fecha**: {date.today().isoformat()}
- **Agente**: A_06_BalanceadorClases
- **Target**: `{TARGET}` (clasificación binaria)
- **Dataset**: tablón completo `05_train_tablon_preseleccion.pkl` ({N_total:,} filas)
- **Split interno**: {TEST_SIZE:.0%} test, estratificado, `random_state={RANDOM_STATE}`
- **Modelo**: regresión logística | **Métrica**: ROC AUC sobre test interno

## Tabla comparativa

{tabla_md}

## Método finalmente elegido

**Ninguno (no balancear).** Los escenarios con balanceo no mejoraron el AUC
del escenario base; se descartan por pérdida de información (undersampling),
aumento de tamaño sin ganancia (oversampling) y complejidad adicional
(SMOTETomek).
"""

# ── Guardado de documentos ───────────────────────────────────────────
ruta_informe = CARPETA_BALANCEO_RES / 'informe_balanceo.md'
ruta_escenarios = CARPETA_BALANCEO_DOC / 'escenarios_balanceo.md'

ruta_informe.write_text(informe_balanceo, encoding='utf-8')
ruta_escenarios.write_text(escenarios_balanceo, encoding='utf-8')

print(f'✅ Documentos guardados:')
print(f'   - {ruta_informe}')
print(f'   - {ruta_escenarios}')


✅ Documentos guardados:
   - c:\Users\Admin\Desktop\Desktop\CarpetaFisica\Data Detective\Ciencia_Datos_2\Caso_Agentes\06_resultados\Balanceo\informe_balanceo.md
   - c:\Users\Admin\Desktop\Desktop\CarpetaFisica\Data Detective\Ciencia_Datos_2\Caso_Agentes\01_Documentos\Balanceo\escenarios_balanceo.md
